# Task 1: Equity Research Assistant

## Assessment objective

Build an end-to-end financial AI notebook that selects a large-cap equity ticker, fetches two years of OHLCV data, computes technical indicators, retrieves recent headlines, runs LLM-backed sentiment and recommendation steps, and renders a concise Markdown/HTML research brief with visible outputs.

## 2. Environment setup check

This cell verifies imports, local package paths, dependency availability, and non-secret LLM configuration. It does not print API keys.

In [1]:
from __future__ import annotations

import json
import math
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "task1_financial":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared.config import configured_llm_providers, get_safe_config_summary
from shared.errors import DataFetchError, LLMProviderError
from shared.finance_data import fetch_ohlcv, fetch_ticker_metadata
from shared.indicators import add_technical_indicators, latest_indicator_snapshot
from shared.news import fetch_headlines
from shared.schemas import EquitySummary, NewsHeadline
from task1_financial.src.report_renderer import write_report_files
from task1_financial.src.sentiment_service import analyze_news_sentiment, build_sentiment_batch_result
from task1_financial.src.signal_reasoner import fallback_recommendation, generate_recommendation

OUTPUT_DIR = PROJECT_ROOT / "task1_financial" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

env_status = {
    "project_root": str(PROJECT_ROOT),
    "python": sys.version.split()[0],
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "configured_llm_providers": configured_llm_providers(),
    "safe_config_summary": get_safe_config_summary(),
    "output_dir": str(OUTPUT_DIR),
}
display(pd.DataFrame([env_status]))

Environment check cell prepared. Run the notebook to refresh live package and provider status.\n

## 3. Ticker selection

The assessment permits AAPL or MSFT. This notebook uses **AAPL** because it is liquid, widely covered, and usually has sufficient price and news history for a reproducible demonstration.

In [2]:
TICKER = "AAPL"
PERIOD = "2y"
MIN_HEADLINES = 10

display(Markdown(f"**Selected ticker:** {TICKER}"))

**Selected ticker:** AAPL

## 4. Fetch 2 years OHLCV

This cell requests two years of daily open, high, low, close, adjusted close, and volume data through the shared yfinance wrapper. If live market data is unavailable in the execution environment, it creates an explicit demo fallback so later notebook sections still show their behavior.

In [3]:
warnings = []

def make_demo_ohlcv(rows: int = 504) -> pd.DataFrame:
    rng = np.random.default_rng(42)
    dates = pd.bdate_range(end=pd.Timestamp.today().normalize(), periods=rows)
    trend = np.linspace(165, 215, rows)
    cycle = 5 * np.sin(np.linspace(0, 10 * np.pi, rows))
    noise = rng.normal(0, 1.8, rows).cumsum() * 0.18
    close = trend + cycle + noise
    open_ = close + rng.normal(0, 1.0, rows)
    high = np.maximum(open_, close) + rng.uniform(0.4, 2.6, rows)
    low = np.minimum(open_, close) - rng.uniform(0.4, 2.6, rows)
    volume = rng.integers(35_000_000, 95_000_000, rows)
    frame = pd.DataFrame(
        {
            "open": open_,
            "high": high,
            "low": low,
            "close": close,
            "adj_close": close,
            "volume": volume,
        },
        index=dates,
    )
    frame.index.name = "date"
    return frame

try:
    prices = fetch_ohlcv(TICKER, period=PERIOD)
    data_source = "live_yfinance"
except Exception as exc:
    prices = make_demo_ohlcv()
    data_source = "demo_fallback"
    warnings.append(f"Live OHLCV fetch failed, using deterministic demo data: {type(exc).__name__}: {exc}")

display(Markdown(f"Fetched **{len(prices):,}** daily rows for **{TICKER}** using `{data_source}`."))
display(prices.tail())

OHLCV fetch cell prepared. Run the notebook to fetch live 2-year AAPL prices or use the explicit demo fallback.\n

## 5. Compute indicators

The indicator layer computes SMA-50, SMA-200, RSI-14, MACD, Bollinger Bands, and a coarse momentum label. Metadata is best-effort because provider coverage varies by ticker and environment.

In [4]:
indicators = add_technical_indicators(prices)
snapshot = latest_indicator_snapshot(indicators)
snapshot.ticker = TICKER
snapshot.as_of = indicators.index[-1].to_pydatetime() if hasattr(indicators.index[-1], "to_pydatetime") else None
snapshot.close_price = float(pd.to_numeric(prices["close"], errors="coerce").dropna().iloc[-1])
snapshot.bollinger_mid = float(indicators["bollinger_middle"].dropna().iloc[-1]) if indicators["bollinger_middle"].notna().any() else None

try:
    metadata = fetch_ticker_metadata(TICKER)
except Exception as exc:
    metadata = {"ticker": TICKER, "company_name": "Apple Inc.", "current_price": snapshot.close_price}
    warnings.append(f"Ticker metadata unavailable: {type(exc).__name__}: {exc}")

equity_summary = EquitySummary(
    ticker=TICKER,
    company_name=metadata.get("company_name") or "Apple Inc.",
    current_price=metadata.get("current_price") or snapshot.close_price,
    week_52_high=metadata.get("week_52_high"),
    week_52_low=metadata.get("week_52_low"),
    pe_ratio=metadata.get("pe_ratio"),
    market_cap=metadata.get("market_cap"),
    sector=metadata.get("sector"),
    indicator_snapshot=snapshot,
)

display(Markdown("Latest technical snapshot:"))
display(pd.DataFrame([snapshot.model_dump(mode="json")]).T.rename(columns={0: "value"}))

Indicator computation cell prepared. Run the notebook to refresh the latest technical snapshot.\n

## 6. Display indicator table

This table shows the latest observations with the computed features so the recommendation inputs are auditable before the LLM step.

In [5]:
indicator_columns = [
    "close",
    "volume",
    "sma_50",
    "sma_200",
    "rsi_14",
    "macd",
    "macd_signal",
    "bollinger_middle",
    "bollinger_upper",
    "bollinger_lower",
    "momentum_signal",
]
indicator_table = indicators[indicator_columns].tail(12).round(2)
display(indicator_table)

Indicator table cell prepared. Run the notebook to display the latest rows.\n

## 7. Fetch news headlines

The news step uses yfinance first and a search fallback second. If neither source is reachable, the notebook records the limitation and supplies a small clearly labeled demo set so the downstream LLM sentiment interface remains visible.

In [6]:
demo_titles = [
    "Apple reports resilient services demand while hardware sales remain mixed",
    "Analysts debate iPhone upgrade cycle ahead of the next product launch",
    "Apple expands AI features across devices as competitors accelerate investment",
    "Supply chain checks point to stable production for flagship devices",
    "Regulatory scrutiny remains an overhang for major app store operators",
]

try:
    news_result = fetch_headlines(TICKER, min_headlines=MIN_HEADLINES, company_name=equity_summary.company_name)
    headlines = news_result.headlines
    warnings.extend(news_result.warnings)
except Exception as exc:
    headlines = []
    warnings.append(f"News fetch failed: {type(exc).__name__}: {exc}")

if not headlines:
    headlines = [NewsHeadline(title=title, ticker=TICKER, source="demo-fallback") for title in demo_titles]
    warnings.append("No live headlines available; using clearly labeled demo headlines for notebook continuity.")

headline_table = pd.DataFrame([h.model_dump(mode="json") for h in headlines])
display(headline_table[["title", "source", "published_at"]].head(MIN_HEADLINES))

News fetch cell prepared. Run the notebook to retrieve fresh headlines.\n

## 8. Run LLM sentiment

Each headline is classified independently using the configured LLM providers. If no provider key is configured or the provider call fails, the notebook returns an empty validated aggregate and records the warning rather than hiding the failure.

In [7]:
try:
    sentiment = analyze_news_sentiment(TICKER, headlines)
except Exception as exc:
    sentiment = build_sentiment_batch_result(TICKER, [])
    warnings.append(f"LLM sentiment failed: {type(exc).__name__}: {exc}")

sentiment_summary = pd.DataFrame([
    {
        "ticker": sentiment.ticker,
        "aggregate_label": sentiment.aggregate_label,
        "aggregate_score": sentiment.aggregate_score,
        "positive_count": sentiment.positive_count,
        "neutral_count": sentiment.neutral_count,
        "negative_count": sentiment.negative_count,
        "classified_headlines": len(sentiment.sentiments),
    }
])
display(sentiment_summary)
if sentiment.sentiments:
    display(pd.DataFrame([item.model_dump(mode="json") for item in sentiment.sentiments]))
else:
    display(Markdown("No validated LLM sentiment rows were produced in this run."))

LLM sentiment cell prepared. Run with GROQ_API_KEY or OPENROUTER_API_KEY for live sentiment.\n

## 9. Run LLM recommendation

The recommendation step combines the technical snapshot and aggregate sentiment into a structured Buy/Hold/Sell view. The project includes a conservative rule fallback so the notebook still produces a validated recommendation if LLM access is unavailable.

In [8]:
try:
    recommendation = generate_recommendation(equity_summary, sentiment)
except Exception as exc:
    recommendation = fallback_recommendation(equity_summary, sentiment)
    warnings.append(f"LLM recommendation failed, used fallback: {type(exc).__name__}: {exc}")

display(pd.DataFrame([recommendation.model_dump(mode="json")]).T.rename(columns={0: "value"}))

LLM recommendation cell prepared. Run the notebook to generate or fallback a recommendation.\n

## 10. Render Markdown/HTML report

This cell writes the assessment deliverables to `task1_financial/outputs`: a Markdown brief, an HTML brief, and the chart image used by the reports.

In [9]:
outputs = write_report_files(equity_summary, sentiment, recommendation, headlines, prices, OUTPUT_DIR)
display(Markdown("### Rendered Markdown preview"))
markdown_text = Path(outputs["markdown"]).read_text(encoding="utf-8") if outputs.get("markdown") else "Markdown report unavailable."
display(Markdown("\n".join(markdown_text.splitlines()[:40])))
display(pd.DataFrame([outputs]))

Report rendering cell prepared. Run the notebook to write Markdown, HTML, and chart files.\n

## 11. Display chart

The chart displays the closing price series used for the technical analysis. It is also saved as a PNG under the output directory.

In [10]:
if outputs.get("chart"):
    display(HTML(f'<img src="{Path(outputs["chart"]).as_posix()}" style="max-width: 100%; height: auto;" alt="{TICKER} price chart">'))
else:
    display(Markdown("Chart output was unavailable."))

Chart display cell prepared. Run the notebook to show the saved chart image.\n

## 12. Show final output paths

These paths are the final notebook artifacts to review or submit with the assessment.

In [11]:
final_output = {
    "notebook": str(PROJECT_ROOT / "task1_financial" / "task1_equity_research.ipynb"),
    "markdown_report": outputs.get("markdown"),
    "html_report": outputs.get("html"),
    "chart_png": outputs.get("chart"),
    "generated_at_utc": datetime.utcnow().isoformat(timespec="seconds"),
}
display(pd.DataFrame([final_output]).T.rename(columns={0: "path_or_value"}))

Final paths cell prepared. Run the notebook to display concrete output artifact paths.\n

## 13. Explain limitations

The final cell makes the notebook's operational and modeling limitations explicit. This is important because the workflow uses external market data, news sources, and LLM inference that can fail, lag, or disagree.

In [12]:
limitations = [
    "This notebook is educational and is not financial advice.",
    "yfinance and news-search data can be delayed, incomplete, revised, or unavailable.",
    "Technical indicators are backward-looking and do not capture fundamentals, valuation changes, or macro shocks.",
    "Headline sentiment is a rough proxy for market narrative and may miss article nuance, source credibility, and timing.",
    "LLM outputs can be sensitive to prompt wording, provider availability, and validation constraints.",
    "Fallback demo data/headlines are clearly labeled and should not be used for investment conclusions.",
]
limitations.extend(warnings)

display(Markdown("### Limitations\n\n" + "\n".join(f"- {item}" for item in limitations)))

### Limitations\n\nRun the notebook to refresh limitations from the current execution.